# Modele de prediction

- **Objectif:** 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    log_loss,
    brier_score_loss
)
from preprocessing import load_and_split, FEATURES, make_preprocessor
from config import PROCESSED_DATA
from sklearn.model_selection import GroupShuffleSplit

## Preparation des donnees

### Chargement des donnees

In [ ]:

# Chargement des donnees
data = pd.read_parquet(PROCESSED_DATA / "longitudinal_panel.parquet")

# Selection du checkpoint J30
data = data[data['checkpoint_day'] == 30].copy()

# Target: abandon apres J30
inscriptions = pd.read_parquet(PROCESSED_DATA / "clean_inscriptions.parquet")

data = data.drop(columns=['target_next_30d']).merge(
    inscriptions[['id_inscription', 'churn']],
    on='id_inscription',
    how='left',
    validate='one_to_one'
)

print(f"Observations: {len(data)}")
print(f"Churn rate: {data['churn'].mean():.4f}")

# Separation des donnees par etudiant
split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=10)
train_val_idx, test_idx = next(split.split(data, groups=data['id_etudiant']))

train_val = data.iloc[train_val_idx]
test = data.iloc[test_idx]

split_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=10)
train_idx, val_idx = next(
    split_val.split(train_val, groups=train_val['id_etudiant'])
)

train = train_val.iloc[train_idx]
val = train_val.iloc[val_idx]

x_tr, y_tr = train[FEATURES], train['churn']
x_val, y_val = val[FEATURES], val['churn']
x_test, y_test = test[FEATURES], test['churn']

### Separation des donnees

In [ ]:
print("Taille totale")
print(len(x_tr) + len(x_val) + len(x_test))
print("taille de training")
print(len(x_tr))
print("taille de validation")
print(len(x_val))
print("taille de test")
print(len(x_test))

print("\nProportion d'abandons")
print(f"Training: {y_tr.mean():.4f}")
print(f"Validation: {y_val.mean():.4f}")
print(f"Test: {y_test.mean():.4f}")

### Transofrmation des donnees

In [ ]:
preprocess = make_preprocessor()

## Modeles de prediction

### Regression logistique

1. Construction

In [ ]:
log_reg = Pipeline([
    ('preprocess', make_preprocessor()),
    ('model', LogisticRegression(max_iter=1000, random_state=10))
])

log_reg.fit(x_tr, y_tr)

2. Evaluation

In [ ]:
y_prob = log_reg.predict_proba(x_val)[:,1]
y_pred = log_reg.predict(x_val)

# Matrice de confusion
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])
disp.plot(cmap='Reds')
plt.title('Matrice de confusion')
plt.show()

print(classification_report(y_val, y_pred, digits=4))
print(f"PR-AUC: {average_precision_score(y_val, y_prob):.4f}")
print(f"Log-Loss: {log_loss(y_val, y_prob, labels=[0,1]):.4f}")
print(f"Brier Score: {brier_score_loss(y_val, y_prob):.4f}")

### Arbre de decision

In [ ]:
tree_model = Pipeline([
    ('preprocess', make_preprocessor(scale_numeric=False)),
    ('model', DecisionTreeClassifier(random_state=10))
])

In [ ]:
tree_model.fit(x_tr, y_tr)
y_prob = tree_model.predict_proba(x_val)[:,1]

print(f"PR-AUC: {average_precision_score(y_val, y_prob):.4f}")
print(f"Log-Loss: {log_loss(y_val, y_prob, labels=[0,1]):.4f}")
print(f"Brier Score: {brier_score_loss(y_val, y_prob):.4f}")

In [ ]:
max_depth = [3, 5, 7, 10, None]
min_leaf = [1, 5, 10, 20]

tree_results = []

for m in max_depth:
    for n in min_leaf:
        model = Pipeline([
            ('preprocess', make_preprocessor(scale_numeric=False)),
            ('model', DecisionTreeClassifier(
                random_state=10,
                max_depth=m,
                min_samples_leaf=n
            ))
        ])

        model.fit(x_tr, y_tr)
        y_prob = model.predict_proba(x_val)[:,1]

        pr_auc = average_precision_score(y_val, y_prob)
        loss = log_loss(y_val, y_prob, labels=[0,1])
        brier = brier_score_loss(y_val, y_prob)

        tree_results.append({
            'max_depth': m,
            'min_samples_leaf': n,
            'PR-AUC': pr_auc,
            'Log-Loss': loss,
            'Brier Score': brier
        })

tree_results = pd.DataFrame(tree_results)
tree_results.sort_values('PR-AUC', ascending=False)

In [ ]:
# Best parameters on validation
best_tree = tree_results.sort_values('PR-AUC', ascending=False).iloc[0]

tree_model = Pipeline([
    ('preprocess', make_preprocessor(scale_numeric=False)),
    ('model', DecisionTreeClassifier(
        random_state=10,
        max_depth=None if pd.isna(best_tree['max_depth']) else int(best_tree['max_depth']),
        min_samples_leaf=int(best_tree['min_samples_leaf'])
    ))
])

tree_model.fit(x_tr, y_tr)

y_prob = tree_model.predict_proba(x_val)[:,1]
y_pred = tree_model.predict(x_val)

print(classification_report(y_val, y_pred, digits=4))
print(f"PR-AUC: {average_precision_score(y_val, y_prob):.4f}")
print(f"Log-Loss: {log_loss(y_val, y_prob, labels=[0,1]):.4f}")
print(f"Brier Score: {brier_score_loss(y_val, y_prob):.4f}")

### SVM

In [ ]:
# Construction
svm = Pipeline([
    ('preprocess', make_preprocessor()),
    ('model', SVC(random_state=10))
])

svm.fit(x_tr, y_tr)

In [ ]:
# Evaluation
y_pred = svm.predict(x_val)
y_score = svm.decision_function(x_val)

print(classification_report(y_val, y_pred, digits=4))
print(f"PR-AUC: {average_precision_score(y_val, y_score):.4f}")

In [ ]:
weighted_models = {
    'Logistic balanced': Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=10
        ))
    ]),
    'Tree balanced': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=False)),
        ('model', DecisionTreeClassifier(
            max_depth=7,
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=10
        ))
    ]),
    'SVM balanced': Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', SVC(
            class_weight='balanced',
            random_state=10
        ))
    ])
}

for name, model in weighted_models.items():
    model.fit(x_tr, y_tr)

    if name == 'SVM balanced':
        scores = model.decision_function(x_val)
    else:
        scores = model.predict_proba(x_val)[:,1]

    y_pred = model.predict(x_val)

    print(f"\n{name}")
    print(f"PR-AUC: {average_precision_score(y_val, scores):.4f}")
    print(f"Recall: {recall_score(y_val, y_pred):.4f}")
    print(f"Precision: {precision_score(y_val, y_pred, zero_division=0):.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

forest = Pipeline([
    ('preprocess', make_preprocessor(scale_numeric=False)),
    ('model', RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=10,
        n_jobs=-1
    ))
])

boosting = Pipeline([
    ('preprocess', make_preprocessor()),
    ('model', HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=10
    ))
])

for name, model in {
    'Random Forest': forest,
    'Gradient Boosting': boosting
}.items():
    model.fit(x_tr, y_tr)
    y_prob = model.predict_proba(x_val)[:,1]

    print(f"\n{name}")
    print(f"PR-AUC: {average_precision_score(y_val, y_prob):.4f}")
    print(f"Log-Loss: {log_loss(y_val, y_prob):.4f}")
    print(f"Brier Score: {brier_score_loss(y_val, y_prob):.4f}")

## Comparaison des modeles

In [ ]:
models = {
    'Regression logistique': log_reg,
    'Arbre de decision': tree_model,
    'SVM': svm,
    'Random Forest': forest,
    'Gradient Boosting': boosting,
    **weighted_models
}

results = []

for name, model in models.items():
    if name in ['SVM', 'SVM balanced']:
        y_score = model.decision_function(x_val)
    else:
        y_score = model.predict_proba(x_val)[:,1]

    results.append({
        'Modele': name,
        'PR-AUC': average_precision_score(y_val, y_score)
    })

results = pd.DataFrame(results)
results.sort_values('PR-AUC', ascending=False)

In [ ]:
# Comparaison avec le modele aleatoire
baseline = y_val.mean()

print(f"PR-AUC baseline: {baseline:.4f}")

sns.barplot(data=results, x='Modele', y='PR-AUC', width=0.8, order=results.sort_values('PR-AUC', ascending=False)['Modele'])
plt.axhline(baseline, color='red', linestyle='--', label='Baseline')
plt.title('Comparaison des modeles')
plt.ylabel('PR-AUC')
plt.legend()
plt.show()

## Evaluation finale

In [ ]:
# Evaluation du Random Forest
y_prob = forest.predict_proba(x_val)[:,1]

from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, y_prob)

threshold_results = pd.DataFrame({
    'threshold': thresholds,
    'precision': precision[:-1],
    'recall': recall[:-1]
})

# Comparaison des seuils de classification
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision_score(y_val, y_pred, zero_division=0):.4f}")
    print(f"Recall: {recall_score(y_val, y_pred):.4f}")

In [ ]:
# Selection du seuil
candidates = threshold_results[threshold_results['precision'] >= 0.30]

if candidates.empty:
    raise ValueError("No threshold satisfies the precision constraint")

best_threshold = candidates.sort_values(
    ['recall', 'precision'],
    ascending=False
).iloc[0]['threshold']

print(f"Selected threshold: {best_threshold:.4f}")

y_pred = (y_prob >= best_threshold).astype(int)
print(classification_report(y_val, y_pred, digits=4))

In [ ]:
from sklearn.base import clone

# Entrainement final
final_model = clone(forest)

x_train = pd.concat([x_tr, x_val])
y_train = pd.concat([y_tr, y_val])

final_model.fit(x_train, y_train)

# Evaluation finale
y_prob = final_model.predict_proba(x_test)[:,1]
y_pred = (y_prob >= best_threshold).astype(int)

print(classification_report(y_test, y_pred, digits=4))
print(f"PR-AUC: {average_precision_score(y_test, y_prob):.4f}")
print(f"Log-Loss: {log_loss(y_test, y_prob):.4f}")
print(f"Brier Score: {brier_score_loss(y_test, y_prob):.4f}")